In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.figsize"] = (20,10)

In [4]:
df1 =pd.read_csv("Mumbai_House_Prices_Updated (1).csv")
df1.head()

,bhk,type,locality,area,price,region,status,age,price_normalized,Distance from Station (km),Number of Amenities,Number of Balconies
0,3,Apartment,Lak And Hanware The Residency Tower,685,2.50,Andheri West,Ready to move,New,250.00,1.872701,1,0
1,2,Apartment,Radheya Sai Enclave Building No 2,640,52.51,Naigaon East,Under Construction,New,52.51,4.753572,4,1
2,2,Apartment,Romell Serene,610,1.73,Borivali West,Under Construction,New,173.00,3.659970,3,1
3,2,Apartment,Soundlines Codename Urban Rainforest,876,59.98,Panvel,Under Construction,New,59.98,2.993292,3,1
4,2,Apartment,Origin Oriana,659,94.11,Mira Road East,Under Construction,New,94.11,0.780093,4,3


In [9]:
df2 = df1.copy()
df2['price_per_sqft'] = df2['price_normalized']*100000/df2['area']
df2.head()

,bhk,type,locality,area,price,region,status,age,price_normalized,Distance from Station (km),Number of Amenities,Number of Balconies,price_per_sqft
0,3,Apartment,Lak And Hanware The Residency Tower,685,2.50,Andheri West,Ready to move,New,250.00,1.872701,1,0,36496.350365
1,2,Apartment,Radheya Sai Enclave Building No 2,640,52.51,Naigaon East,Under Construction,New,52.51,4.753572,4,1,8204.687500
2,2,Apartment,Romell Serene,610,1.73,Borivali West,Under Construction,New,173.00,3.659970,3,1,28360.655738
3,2,Apartment,Soundlines Codename Urban Rainforest,876,59.98,Panvel,Under Construction,New,59.98,2.993292,3,1,6847.031963
4,2,Apartment,Origin Oriana,659,94.11,Mira Road East,Under Construction,New,94.11,0.780093,4,3,14280.728376


In [8]:
len(df2.region.unique())

228

In [10]:
df2.to_csv('output.csv')  

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
df = pd.read_csv("Mumbai_House_Prices_Updated_v2.csv")

# Select relevant features and target variable
features = ["area", "bhk", "Distance from Station (km)", "Number of Amenities", "Number of Balconies"]
target = "price"

# Drop missing values
df = df.dropna(subset=features + [target])

# Split the dataset into training and testing sets
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = rf_model.predict(X_test_scaled)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

# Print evaluation metrics
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {r2}")

Mean Absolute Error: 21.693846929247762
Root Mean Squared Error: 29.59922760680828
R2 Score: 0.19384617839100038


In [13]:
new_house = np.array([[750, 2, 1.5, 3, 2]])  # (area, bhk, distance, amenities, balconies)

# Scale the input using the same scaler used for training
new_house_scaled = scaler.transform(new_house)

# Predict the price
predicted_price = rf_model.predict(new_house_scaled)

print(f"Predicted Price: {predicted_price[0]}")

Predicted Price: 10.60460000000001


C:\Users\Taqueveem Ahmad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [15]:
import numpy as np
import pandas as pd

# Example input for a house
new_house = np.array([[100, 1, 5, 1, 0]])  # (area, bhk, distance, amenities, balconies)

# Convert to DataFrame with correct column names
new_house_df = pd.DataFrame(new_house, columns=["area", "bhk", "Distance from Station (km)", "Number of Amenities", "Number of Balconies"])

# Scale the input
new_house_scaled = scaler.transform(new_house_df)

# Predict the price
predicted_price = rf_model.predict(new_house_scaled)

print(f"Predicted Price: {predicted_price[0]}")


Predicted Price: 52.228899999999996


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
file_path = "Mumbai_House_Prices_Updated_v2.csv"
df = pd.read_csv(file_path)

# Define price adjustment rules
base_price = 25  # Base price in lakhs
df["price"] = (base_price * (df["area"] / 100) +
               df["bhk"] * 5 +
               df["Number of Amenities"] * 3 -
               df["Distance from Station (km)"] * 2 +
               df["Number of Balconies"] * 2)

# Increase price for apartments
if "Type" in df.columns:
    df.loc[df["Type"] == "Apartment", "price"] *= 1.2

# Define features and target
features = ["area", "bhk", "Distance from Station (km)", "Number of Amenities", "Number of Balconies"]
X = df[features]
y = df["price"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Save the model and scaler
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "features.pkl")

print("Model and scaler saved successfully!")


Mean Absolute Error: 0.77 lakhs
Root Mean Squared Error: 1.39 lakhs
R² Score: 1.00
Predicted Price: 116.07 lakhs


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
file_path = "Mumbai_House_Prices_Updated_v2.csv"
df = pd.read_csv(file_path)

# Define price adjustment rules
base_price = 25  # Base price in lakhs
df["price"] = (base_price * (df["area"] / 100) +
               df["bhk"] * 5 +
               df["Number of Amenities"] * 3 -
               df["Distance from Station (km)"] * 2 +
               df["Number of Balconies"] * 2)

# Increase price for apartments
if "Type" in df.columns:
    df.loc[df["Type"] == "Apartment", "price"] *= 1.2

# Define features and target
features = ["area", "bhk", "Distance from Station (km)", "Number of Amenities", "Number of Balconies"]
X = df[features]
y = df["price"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Save the model and scaler
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "features.pkl")

print("Model and scaler saved successfully!")


Mean Absolute Error: 0.77 lakhs
Root Mean Squared Error: 1.39 lakhs
R² Score: 1.00
Predicted Price: 116.07 lakhs


In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
file_path = "Mumbai_House_Prices_Updated_v2.csv"
df = pd.read_csv(file_path)

# Define price adjustment rules
base_price = 25  # Base price in lakhs
df["price"] = (base_price * (df["area"] / 100) +
               df["bhk"] * 5 +
               df["Number of Amenities"] * 3 -
               df["Distance from Station (km)"] * 2 +
               df["Number of Balconies"] * 2)

# Increase price for apartments
if "Type" in df.columns:
    df.loc[df["Type"] == "Apartment", "price"] *= 1.2

# Define features and target
features = ["area", "bhk", "Distance from Station (km)", "Number of Amenities", "Number of Balconies"]
X = df[features]
y = df["price"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Save the model and scaler
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "features.pkl")

print("Model and scaler saved successfully!")



Model and scaler saved successfully!
